# Polyp segmentation: one complete Kaggle notebook

**Before you press run (once):** Settings -> **Accelerator: GPU T4 x2**, **Internet: ON**. **+ Add Input** -> *Medico automatic polyp segmentation dataset* (by Debesh Jha) **and** the **Output of your last run** (Notebook Output Files). For the PolypGen test: **Add-ons -> Secrets -> add a secret named `SYNAPSE_TOKEN`** (your Synapse personal access token, see the guide). PolypGen is downloaded INSIDE the session into a temporary folder that is not saved; it is never uploaded anywhere.

**Then:** **Save Version -> Save & Run All (Commit)**. The notebook does, in order:
1. install + download the code, build the data, safety check (must say `SELF-CHECK PASSED`)
2. **Stage 1:** train (if switched on) and evaluate Protocol A (Kvasir-880 only, zero-shot on 5 test sets) for every model version
3. **Stage 1c/1d:** extra evaluations of finished models without training: PraNet metric, multi-scale test, accuracy frontier + mask clean-up, zoom cascade, size counterfactuals
4. **Stage 1e / 2:** PolypGen zero-shot test (frozen models only); the standard protocols B and Bclean
5. print all numbers, make all figures (`figures/*.png` and `key_numbers.md`), make `results_bundle.zip`

**If it stops** (12-hour limit, a crash, a disconnect): start a new **Save & Run All** and add the previous version's output as an input. The notebook finds it by itself and **skips everything already finished**.

In [ ]:
# ===================== SETTINGS (normally you change nothing) =====================
GIT_URL  = "https://github.com/KUNALSHAWW/Transformer-Based-Robust-Polyp-Segmentation-for-Cross-Dataset-Colonoscopy-Analysis.git"
BRANCH   = "paper-research"
# Protocol A versions, trained in this order (the first five were finished earlier and are skipped when a previous output is attached)
PLAN_A   = ["base", "roy", "roy_size", "roy_overlay", "roy_full",
            "base_ms", "base_512",                                   # (finished earlier, skipped when the earlier output is attached)
            "base_b2", "base_b1", "base_b0",
            "base_b2_kd", "base_b1_kd", "base_b0_kd",
            "roy_b1", "roy_b1_kd", "roy_b0",
            "base_zoom_match", "base_zoom", "base_zoom_black",       # NEW: scale-matched zoom-out, zoom-out on a natural background, control with the old black canvas
            "base_pid",                                              # NEW: per-image Dice instead of the batch-pooled Dice
            "base_lab", "base_lab_zoom"]                             # colour exchange (SANet), colour exchange + zoom-out
VARIANT_SEEDS = {"base": [0, 1, 2]}   # versions that get more seeds (to measure seed-to-seed noise); all others use SEEDS
RUN_B_AND_BCLEAN = True    # afterwards: the standard protocols B and Bclean for the versions in B_VARIANTS
B_VARIANTS = ["base"]   # B base is already evaluated in the earlier output; Bclean base is new. [] = base plus the best Protocol-A version
SAVE_PRED_FOR = ["base", "roy_full", "base_ms", "base_512", "base_b2", "base_b2_kd", "base_b1_kd", "base_zoom", "base_zoom_match"]   # save predicted masks of these versions for the picture figures
SEEDS    = [0]
RUN_TRAIN = True           # False = evaluate-only session (no training at all; use it when few GPU hours are left)
EVAL_A    = True           # False = skip the standard evaluation of Protocol A (the extra evaluations below still run)
RESUME_COPY = True         # False = do not copy the old runs into /kaggle/working (read the checkpoints from the attached input; evaluate-only sessions)
# extra evaluations of finished models (no training). est_min = rough minutes on a T4; skipped if they would start after EVAL_DEADLINE_H
EXTRA_EVAL = [
    dict(tag="base_pranet_fb", variant="base", est_min=10, args="--pranet_metric --fallback 0.5"),                       # PraNet-toolbox metric + 'never empty' fallback
    dict(tag="base_ms_tta", variant="base", est_min=24, args="--pranet_metric --fallback 0.5 --scales 1.0,1.25,1.5"),    # multi-scale test-time augmentation
    dict(tag="ens_base_roy_full", variant="base", est_min=20, args="--pranet_metric --ens_with roy_full --thr 0.5"),     # ensemble of two different versions
]
# extra evaluations without training (experiments of evaluate_extra.py). est_min = rough minutes on a T4. Each experiment is first tried on 3 pictures; if that fails it is skipped.
XEVAL = [
    dict(exp="views",    variant="base", est_min=30),   # accuracy frontier (1 model / flips / ensemble / both) + 5 ways to clean up the final mask
    dict(exp="zoom",     variant="base", est_min=25),   # zoom cascade: look again around the strongest peaks where the prediction is small or empty
    dict(exp="sizeswap", variant="base", est_min=25),   # size counterfactuals: shrink Kvasir pictures, oracle-enlarge ColonDB / ETIS pictures
    dict(exp="sizeswap", variant="base_zoom_match", est_min=25),   # same, for the scale-matched model (only if trained)
    dict(exp="sizeswap", variant="base_zoom", est_min=25),
]
# PolypGen (6 hospitals + polyp-free frames) = untouched confirmation data. ONLY versions that are frozen may be listed here; do not add versions after looking at results.
RUN_POLYPGEN = True
POLYPGEN_VARIANTS = [("A", "base"), ("B", "base")]          # (protocol the model was trained under, version)
POLYPGEN_ID = "syn45200214"                                  # Synapse folder of the dataset (project syn26376615)
EPOCHS   = 60              # maximum; early stopping (patience 12, not before epoch 10) usually ends a fold near epoch 35-40
WORKERS  = 2
BS       = 8               # use 4 if you see "out of memory"
TIME_LIMIT_H = 9.0         # do not start a new fold if the session would run past this many hours (then just run the notebook again)
EVAL_DEADLINE_H = 10.3     # after this many hours no further evaluation is started (it is done in the next run); Kaggle saves nothing if a run passes 12 h or your weekly GPU hours run out
REQUIRE_RESUME = True      # stop at once if no previous output is attached (set False only for the very first run ever)
# ---- developer-only, leave as they are on Kaggle ----
CODE_OVERRIDE, DATA_OVERRIDE = "", ""
WORK, INPUT_ROOT, SMOKE = "/kaggle/working", "/kaggle/input", False
# ==================================================================================

In [ ]:
import os, sys, glob, json, time, shutil, subprocess, zipfile
T0 = time.time()
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
sh("pip install -q segmentation-models-pytorch albumentations timm")
if CODE_OVERRIDE:
    CODE = CODE_OVERRIDE
else:
    if os.path.exists(f"{WORK}/code"): shutil.rmtree(f"{WORK}/code")
    sh(f"git clone -q --depth 1 -b {BRANCH} {GIT_URL} {WORK}/code")
    CODE = f"{WORK}/code/polyp_paper"
    sh(f"git -C {WORK}/code log -1 --format='code version: %h %s'")
sys.path.insert(0, CODE)
import torch
NGPU = torch.cuda.device_count()
print("GPUs visible:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
RUNS, EVAL = f"{WORK}/runs", f"{WORK}/eval"; os.makedirs(RUNS, exist_ok=True); os.makedirs(EVAL, exist_ok=True)

In [ ]:
# ---- resume: copy finished work from any previous output that is attached as an input ----
n_copied = 0
for src in glob.glob(f"{INPUT_ROOT}/**/runs", recursive=True):
    if os.path.realpath(src) == os.path.realpath(RUNS): continue
    if RESUME_COPY: shutil.copytree(src, RUNS, dirs_exist_ok=True)
    else: RUNS = src                                          # evaluate-only: read the checkpoints straight from the attached input
    n_copied += 1
    ev = os.path.join(os.path.dirname(src), "eval")
    if os.path.isdir(ev): shutil.copytree(ev, EVAL, dirs_exist_ok=True)
print("resumed from", n_copied, "previous output(s)" if n_copied else "nothing (fresh start)")
if REQUIRE_RESUME and not n_copied and not SMOKE:
    raise SystemExit("STOP: no previous output found under " + INPUT_ROOT + ". Add the earlier run as an input (+ Add Input -> Notebook Output Files) and run again, or set REQUIRE_RESUME = False in the settings cell if this really is the very first run.")

# ---- data: reuse a ready data folder if one exists, otherwise build it (about 5-15 minutes) ----
if DATA_OVERRIDE:
    DATA = DATA_OVERRIDE
else:
    ready = glob.glob(f"{INPUT_ROOT}/**/data/splits/clinicdb_sequences.csv", recursive=True)
    DATA = os.path.dirname(os.path.dirname(ready[0])) if ready else f"{WORK}/data"
    if not os.path.exists(DATA + "/splits/clinicdb_sequences.csv"):
        cands = [c for c in glob.glob(f"{INPUT_ROOT}/**/Kvasir-SEG", recursive=True) if os.path.isdir(os.path.join(c, "images"))]
        assert cands, "Kvasir-SEG not found: use + Add Input -> 'Medico automatic polyp segmentation dataset' (by Debesh Jha)"
        sh(f'{sys.executable} {CODE}/prepare_data.py --out {DATA} --kvasir "{cands[0]}" --download')
        shutil.rmtree(DATA + "/_dl", ignore_errors=True)          # free about 1.5 GB of downloaded zip files
print("data folder:", DATA, os.listdir(DATA))
for p in ("A", "B", "Bclean"):
    sh(f"cd {CODE} && {sys.executable} splits.py --data_root {DATA} --protocol {p} | head -4")

In [ ]:
# ---- safety check of the code (about 1 minute). It must end with: SELF-CHECK PASSED ----
sh(f"cd {CODE} && {sys.executable} selfcheck.py --data_root {DATA} 2>&1 | grep -v -i -E 'warn|deprecat|original_init' ")

In [ ]:
# ---- size and speed of every model version (parameters, GFLOPs, milliseconds per picture on this machine) ----
os.makedirs(f"{WORK}/profile", exist_ok=True)
if RUN_TRAIN: sh(f"cd {CODE} && {sys.executable} profile_models.py --variants {' '.join(PLAN_A)} --out {WORK}/profile/profile.csv 2>&1 | grep -v -i -E 'warn|deprecat|original_init' | tail -20")

In [ ]:
# ---- helper functions (stages below use them) ----
fold_done = lambda p, v, s, f: os.path.exists(f"{RUNS}/{p}/{v}/seed{s}/fold{f}/manifest.json")
seeds_of = lambda v: VARIANT_SEEDS.get(v, SEEDS)
jobs_for = lambda p, variants: [(p, v, s, f) for v in variants for s in seeds_of(v) for f in range(5) if not fold_done(p, v, s, f)]
EST_S = {"base": 32, "roy": 32, "roy_size": 33, "roy_overlay": 59, "roy_full": 56,          # measured seconds per epoch on a T4
         "base_ms": 34, "base_512": 58, "base_b2": 26, "base_b1": 22, "base_b0": 18, "roy_b1": 22, "roy_b0": 18,
         "base_b2_kd": 34, "base_b1_kd": 30, "base_b0_kd": 26, "roy_b1_kd": 30,                # estimates for the new versions (safe side)
         "base_lab": 36, "base_zoom": 36, "base_lab_zoom": 38, "base_zoom_black": 36, "base_zoom_match": 36, "base_pid": 33}
est_seconds = lambda p, v: EST_S.get(v, 60) * 45 * (1.65 if p in ("B", "Bclean") else 1.0) * 1.1   # generous estimate for one fold
SMOKE_ARGS = ["--dry", "4", "--epochs", "2", "--no_pretrained", "--workers", "0"] if SMOKE else []

def tail(path, n=1):
    try: return open(path).read().strip().split("\n")[-n:]
    except Exception: return [""]

def run_jobs(jobs):
    """run (protocol, variant, seed, fold) jobs, one per GPU at a time; retry a failed fold once; honour the time budget."""
    queue, running, tries, stop, tick = list(jobs), {}, {}, False, 0
    if not RUN_TRAIN: print("RUN_TRAIN is False: no training in this session (", len(queue), "fold(s) would be left)"); return
    print(len(queue), "fold(s) to train")
    while queue or running:
        for g in range(max(1, NGPU)):
            if g in running or not queue or stop: continue
            p, v, s, f = queue[0]
            if time.time() - T0 + est_seconds(p, v) > TIME_LIMIT_H * 3600:
                print(f"time budget ({TIME_LIMIT_H} h) reached: no new folds will start. Run the notebook again to continue."); stop = True; queue.clear(); break
            queue.pop(0); log = f"{RUNS}/log_{p}_{v}_s{s}_f{f}.txt"
            cmd = [sys.executable, "train.py", "--data_root", DATA, "--protocol", p, "--variant", v, "--fold", str(f), "--seed", str(s),
                   "--out", RUNS, "--epochs", str(EPOCHS), "--workers", str(WORKERS), "--bs", str(BS)] + SMOKE_ARGS
            proc = subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)), stdout=open(log, "w"), stderr=subprocess.STDOUT)
            running[g] = (proc, (p, v, s, f), time.time(), log); print(f"[{(time.time()-T0)/60:6.1f} min] start  {p}/{v} seed{s} fold{f} on GPU {g}", flush=True)
        time.sleep(15); tick += 1
        for g, (proc, job, t0, log) in list(running.items()):
            if proc.poll() is None: continue
            del running[g]; ok = proc.returncode == 0
            print(f"[{(time.time()-T0)/60:6.1f} min] {'OK    ' if ok else 'FAILED'} {job} ({(time.time()-t0)/60:.1f} min)", flush=True)
            if not ok:
                print("   last log lines:", tail(log, 4)); tries[job] = tries.get(job, 0) + 1
                if tries[job] < 2 and not stop: queue.insert(0, job); print("   retrying once")
        if tick % 40 == 0:
            for g, (proc, job, t0, log) in running.items(): print("   progress:", job, tail(log)[0][:110], flush=True)

def evaluate_done(p, variants):
    for v in variants:
        for s in seeds_of(v):
            save = v in SAVE_PRED_FOR
            if os.path.exists(f"{EVAL}/{p}/{v}/seed{s}/summary.json") and not (save and not os.path.isdir(f"{EVAL}/{p}/{v}/seed{s}/preds")):
                print("already evaluated:", p, v, s); continue
            if not all(fold_done(p, v, s, f) for f in range(5)): print("skip (not all 5 folds finished):", p, v, s); continue
            if time.time() - T0 > EVAL_DEADLINE_H * 3600: print(f"evaluation deadline ({EVAL_DEADLINE_H} h) passed: not evaluating {p} {v} now; run the notebook again"); continue
            sh(f"cd {CODE} && {sys.executable} evaluate.py --data_root {DATA} --protocol {p} --variant {v} --seed {s} --runs {RUNS} --out {EVAL} --overlay_eval"
               + (" --save_pred" if save else "") + (" --limit 2" if SMOKE else "") + " 2>&1 | grep -v -i -E 'warn|deprecat|original_init'")

def best_of(variants, protocol="A"):
    """best version by the mean best INNER-validation Dice over its folds (training-side only; test sets are never used for this choice)."""
    sc = {}
    for v in variants:
        d = [json.load(open(f"{RUNS}/{protocol}/{v}/seed{s}/fold{f}/manifest.json"))["best_val_dice"] for s in seeds_of(v) for f in range(5) if fold_done(protocol, v, s, f)]
        if len(d) == 5 * len(seeds_of(v)): sc[v] = sum(d) / len(d)
    print("inner-validation Dice of finished versions:", {k: round(x, 4) for k, x in sc.items()})
    return max(sc, key=sc.get) if sc else None
print("helpers ready")

In [ ]:
# ===== STAGE 1a: train Protocol A (train on Kvasir-880 only) =====
run_jobs(jobs_for("A", PLAN_A))

In [ ]:
# ===== STAGE 1b: evaluate Protocol A on the five test sets (threshold chosen from out-of-fold predictions only) =====
if EVAL_A: evaluate_done("A", PLAN_A)

In [ ]:
# ===== STAGE 1c: extra evaluations of finished models (PraNet-toolbox metric, never-empty fallback, multi-scale test, ensembles) =====
EVAL_X = f"{WORK}/eval_extra"; os.makedirs(EVAL_X, exist_ok=True)
for e in EXTRA_EVAL:
    out_json = f"{EVAL_X}/A/{e['tag']}/seed0/summary.json"
    if os.path.exists(out_json): print("already done:", e["tag"]); continue
    if not all(fold_done("A", e["variant"], 0, f) for f in range(5)): print("skip (folds missing):", e["tag"]); continue
    if time.time() - T0 + e["est_min"] * 60 > EVAL_DEADLINE_H * 3600: print("skip (not enough time left):", e["tag"]); continue
    sh(f"cd {CODE} && {sys.executable} evaluate.py --data_root {DATA} --protocol A --variant {e['variant']} --seed 0 --runs {RUNS} --out {EVAL_X} --tag {e['tag']} {e['args']}"
       + (" --limit 2" if SMOKE else "") + " 2>&1 | grep -v -i -E 'warn|deprecat|original_init'")

In [ ]:
# ===== STAGE 1d: more evaluations of finished models (no training) =====
EVAL_X = f"{WORK}/eval_x"; os.makedirs(EVAL_X, exist_ok=True)
def run_ok(cmd):
    """run a command, show its last lines, return True if it succeeded (the real exit code of the script, not of a filter)"""
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = [l for l in r.stdout.splitlines() if not any(w in l.lower() for w in ("warn", "deprecat", "original_init"))]
    print("\n".join(lines[-14:] if r.returncode == 0 else lines[-25:]), flush=True); return r.returncode == 0
smoke_ok = {}
def xrun(e, extra="", protocol="A", seed=0):
    tag = f"{protocol}_{e['variant']}_s{seed}"
    if os.path.exists(f"{EVAL_X}/{e['exp']}/{tag}/meta.json"): print("already done:", e["exp"], tag); return
    if not all(fold_done(protocol, e["variant"], seed, f) for f in range(5)): print("skip (folds missing):", e["exp"], tag); return
    if time.time() - T0 + e["est_min"] * 60 > EVAL_DEADLINE_H * 3600: print("skip (not enough time left):", e["exp"], tag); return
    base = f"cd {CODE} && {sys.executable} evaluate_extra.py --exp {e['exp']} --data_root {DATA} --protocol {protocol} --variant {e['variant']} --seed {seed} --runs {RUNS} {extra}"
    if e["exp"] not in smoke_ok:      # 3-picture trial first: a bug shows up in a minute instead of after an hour
        smoke_ok[e["exp"]] = run_ok(base + f" --out {WORK}/smoke_x --limit 3")
        print("trial of", e["exp"], "OK" if smoke_ok[e["exp"]] else "FAILED -> this experiment is skipped")
    if smoke_ok[e["exp"]]:
        print(">>>", e["exp"], tag, flush=True); run_ok(base + f" --out {EVAL_X}")
for e in XEVAL: xrun(e)

In [ ]:
# ===== STAGE 1e: PolypGen, zero-shot, FROZEN versions only (downloaded inside this session, never saved, never uploaded) =====
# Data terms: research use only, no redistribution. Cite: Ali et al. Sci Data 2023 (10:75); Ali et al. Sci Rep 2024 (14:2032); Ali et al. Med Image Anal 2021 (102002). Dataset DOI 10.7303/syn26376615.
PG_DIR = "/kaggle/temp/polypgen" if os.path.isdir("/kaggle/temp") else "/tmp/polypgen"
assert not os.path.realpath(PG_DIR).startswith(os.path.realpath(WORK)), "PolypGen must never be inside the saved output folder"
tok = None
if RUN_POLYPGEN:
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("SYNAPSE_TOKEN")
    except Exception as ex:
        print("PolypGen skipped: no secret SYNAPSE_TOKEN found (", type(ex).__name__, "). Add it under Add-ons -> Secrets, then run again.")
if tok:
    todo = [(p_, v_) for p_, v_ in POLYPGEN_VARIANTS if not os.path.exists(f"{EVAL_X}/polypgen/{p_}_{v_}_s0/meta.json") and all(fold_done(p_, v_, 0, f) for f in range(5))]
    if todo and time.time() - T0 + 45 * 60 * len(todo) < EVAL_DEADLINE_H * 3600:
        sh("pip install -q synapseclient")
        import polypgen
        PG_ROOT = polypgen.download(tok, PG_DIR); del tok                    # the token is never printed
        its = polypgen.items(PG_ROOT); import collections; print(len(its), "PolypGen images:", dict(collections.Counter(i["kind"] for i in its)))
        for p_, v_ in todo: xrun(dict(exp="polypgen", variant=v_, est_min=45), extra=f"--polypgen_root {PG_ROOT}", protocol=p_)
        shutil.rmtree(PG_DIR, ignore_errors=True); print("PolypGen data deleted from this session")
    else: print("PolypGen: nothing to do or not enough time left (", todo, ")")

In [ ]:
# ===== STAGE 2a: choose the best Protocol-A version, then train the standard protocols B and Bclean =====
BEST = best_of(PLAN_A)
VARS_B = list(B_VARIANTS) if B_VARIANTS else list(dict.fromkeys(["base"] + ([BEST] if BEST else [])))
print("best version (training-side score):", BEST, "-> standard protocols for", VARS_B)
if RUN_B_AND_BCLEAN and VARS_B:
    run_jobs(jobs_for("B", VARS_B) + jobs_for("Bclean", VARS_B))

In [ ]:
# ===== STAGE 2b: evaluate B and Bclean =====
if RUN_B_AND_BCLEAN and VARS_B:
    evaluate_done("B", VARS_B); evaluate_done("Bclean", VARS_B)

In [ ]:
# ===== SUMMARY: one table to look at (mean over test images) and the training speed =====
rows = []
for f in sorted(glob.glob(f"{EVAL}/*/*/seed*/summary.json")):
    p, v, s = f.replace(os.sep, "/").split("/")[-4:-1]; sm = json.load(open(f))["summary"]
    for d, m in sm.items(): rows.append((p, v, s, d, m["n"], m["dice"], m["iou"], m["hd95"]))
print(f"{'protocol':8s} {'version':16s} {'seed':6s} {'test set':20s} {'n':>4s} {'Dice':>7s} {'IoU':>7s} {'HD95':>8s}")
for r in rows: print(f"{r[0]:8s} {r[1]:16s} {r[2]:6s} {r[3]:20s} {r[4]:4d} {r[5]:7.4f} {r[6]:7.4f} {r[7]:8.1f}")
print()
print("EXTRA EVALUATIONS (Protocol A): fixed-threshold Dice | PraNet-toolbox Dice | Dice with never-empty fallback | HD95 | HD95 with fallback | share of empty masks")
for f in sorted(glob.glob(f"{WORK}/eval_extra/*/*/seed*/summary.json")):
    tag = f.replace(os.sep, "/").split("/")[-3]
    for d, m in json.load(open(f))["summary"].items():
        if "+overlay" in d: continue
        print(f"  {tag:20s} {d:12s} {m['dice']:.4f} | {m.get('pn_dice', float('nan')):.4f} | {m.get('fb_dice', float('nan')):.4f} | {m['hd95']:.1f} | {m.get('fb_hd95', float('nan')):.1f} | {m.get('was_empty', float('nan')):.3f}")
print()
for f in sorted(glob.glob(f"{RUNS}/*/*/seed0/fold0/history.json")):
    p, v = f.replace(os.sep, "/").split("/")[-5:-3]; h = json.load(open(f))
    print(f"speed {p}/{v}: {h[-1]['sec']:.0f} s/epoch (fold 0 ran {len(h)} epochs, best validation Dice {max(e['val_dice'] for e in h):.4f})")
print("total wall time of this session: %.1f h" % ((time.time() - T0) / 3600))
du = shutil.disk_usage(WORK); print("disk used in", WORK, ": %.1f GB of %.1f GB" % ((du.total - du.free) / 1e9, du.total / 1e9))

In [ ]:
# ===== all figures + one text file of key numbers (so you can look at everything without anybody generating pictures) =====
FIG = f"{WORK}/figures"; os.makedirs(FIG, exist_ok=True)
run_ok(f"cd {CODE} && {sys.executable} make_figures.py --bundle {WORK} --out {FIG} --data_root {DATA}")
print(open(f"{FIG}/key_numbers.md").read() if os.path.exists(f"{FIG}/key_numbers.md") else "no key numbers file")
try:                                  # show the figures right here in the notebook as well
    from IPython.display import Image, display
    for f in sorted(glob.glob(f"{FIG}/fig_*.png")): print(os.path.basename(f)); display(Image(f))
except Exception as ex: print("could not display:", ex)

In [ ]:
# ===== package the small result files (no model weights, no large prediction files, NO PolypGen pictures) =====
bundle = f"{WORK}/results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for root in (RUNS, EVAL, f"{WORK}/eval_extra", f"{WORK}/eval_x", f"{WORK}/profile", f"{WORK}/figures"):
        for dp, _, fs in os.walk(root):
            for f in fs:
                if f.endswith(".pt") or f == "oof_probs.npz": continue
                p = os.path.join(dp, f); z.write(p, os.path.relpath(p, WORK))
print(bundle, round(os.path.getsize(bundle) / 1e6, 1), "MB  <- download this file from the Output tab and keep it with your logbook")